# ETL Pipeline — Main Orchestration

This notebook orchestrates the end-to-end ETL pipeline:

1. **Extract** — Reads raw CSV data from a Unity Catalog volume
2. **Transform** — Cleans, validates, and enriches the data
3. **Load** — Writes transformed data to a Delta table

**Usage**: Run this notebook directly or trigger it via a Databricks Job.

**Parameters** (via widgets):
- `source_path` — Path to the source CSV file
- `catalog` — Target Unity Catalog catalog
- `schema` — Target schema
- `table_name` — Target Delta table name

In [0]:
# Databricks notebook source
# --- Main: Define pipeline widgets / parameters ---

import time

# Define widgets for job parameters
SOURCE_PATH_DEFAULT = "/Volumes/workspace/default/etl_source_data/sales_data.csv"
CATALOG_DEFAULT = "workspace"
SCHEMA_DEFAULT = "default"
TABLE_NAME_DEFAULT = "sales_delta"

# Create widgets
widgets = dict(dbutils.widgets.getAll())

if "source_path" not in widgets:
    dbutils.widgets.text("source_path", SOURCE_PATH_DEFAULT)
if "catalog" not in widgets:
    dbutils.widgets.text("catalog", CATALOG_DEFAULT)
if "schema" not in widgets:
    dbutils.widgets.text("schema", SCHEMA_DEFAULT)
if "table_name" not in widgets:
    dbutils.widgets.text("table_name", TABLE_NAME_DEFAULT)

# Refresh widget values
widgets = dict(dbutils.widgets.getAll())

print("=" * 60)
print("ETL Pipeline — Starting")
print("=" * 60)
print(f"Source: {widgets.get('source_path', SOURCE_PATH_DEFAULT)}")
print(f"Target: {widgets.get('catalog', CATALOG_DEFAULT)}.{widgets.get('schema', SCHEMA_DEFAULT)}.{widgets.get('table_name', TABLE_NAME_DEFAULT)}")
print("=" * 60)


In [0]:
# Databricks notebook source
# --- Step 1: Extract ---

print("\n[1/3] Extracting data...")
start_time = time.time()

# --- Extract: Read raw sales data from UC volume ---
_source_path = widgets.get("source_path", "/Volumes/workspace/default/etl_source_data/sales_data.csv")
print(f"Reading CSV from: {_source_path}")

raw_sales_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(_source_path)
)

row_count = raw_sales_df.count()
print(f"Extracted {row_count} rows")
raw_sales_df.printSchema()
raw_sales_df.show(5, truncate=False)

raw_sales_df.createOrReplaceTempView("raw_sales")
print("Extract step completed successfully.")

extract_time = time.time() - start_time
print(f"\nExtraction completed in {extract_time:.2f}s")


In [0]:
# Databricks notebook source
# --- Step 2: Transform ---

print("\n[2/3] Transforming data...")
start_time = time.time()

# --- Transform: Clean, enrich, and validate sales data ---
from pyspark.sql.functions import col, trim, upper, month, year, date_format, when, sum as _sum

raw_sales_df = spark.table("raw_sales")

deduped_df = raw_sales_df.distinct()
print(f"After deduplication: {deduped_df.count()} rows (was {raw_sales_df.count()})")

cleaned_df = (
    deduped_df
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("region", upper(trim(col("region"))))
)

valid_df = cleaned_df.filter(
    col("order_id").isNotNull()
    & (col("quantity") > 0)
    & (col("unit_price") > 0)
)
removed_count = cleaned_df.count() - valid_df.count()
print(f"Removed {removed_count} invalid records")

transformed_sales_df = (
    valid_df
    .withColumn("total_amount", (col("quantity") * col("unit_price")).cast("decimal(10,2)"))
    .withColumn("order_month", month(col("order_date")))
    .withColumn("order_year", year(col("order_date")))
    .withColumn("order_date_str", date_format(col("order_date"), "yyyy-MM-dd"))
)

print(f"\nTransformed row count: {transformed_sales_df.count()}")
print("\nSales by region:")
transformed_sales_df.groupBy("region").agg(
    _sum("total_amount").alias("total_revenue"),
    col("region")
).orderBy(col("total_revenue").desc()).show()

print("\nSales by category:")
transformed_sales_df.groupBy("category").agg(
    _sum("total_amount").alias("total_revenue")
).orderBy(col("total_revenue").desc()).show()

print("\nSample transformed data:")
transformed_sales_df.show(5, truncate=False)

transformed_sales_df.createOrReplaceTempView("transformed_sales")
print("Transform step completed successfully.")

transform_time = time.time() - start_time
print(f"\nTransformation completed in {transform_time:.2f}s")


In [0]:
# Databricks notebook source
# --- Step 3: Load ---

print("\n[3/3] Loading data...")
start_time = time.time()

# --- Load: Write transformed data to Delta table ---
_catalog = widgets.get("catalog", "workspace")
_schema = widgets.get("schema", "default")
_table_name = widgets.get("table_name", "sales_delta")
full_table_name = f"{_catalog}.{_schema}.{_table_name}"
print(f"Writing to table: {full_table_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {_catalog}.{_schema}")

transformed_sales_df = spark.table("transformed_sales")

(
    transformed_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"\nSuccessfully wrote {transformed_sales_df.count()} rows to {full_table_name}")

result = spark.table(full_table_name)
print(f"\nVerification — row count in table: {result.count()}")
result.printSchema()
result.show(5, truncate=False)
print("Load step completed successfully.")

load_time = time.time() - start_time
print(f"\nLoading completed in {load_time:.2f}s")

# Final summary
print("\n" + "=" * 60)
print("ETL Pipeline — Completed Successfully")
print("=" * 60)
print(f"  Extract time:   {extract_time:.2f}s")
print(f"  Transform time: {transform_time:.2f}s")
print(f"  Load time:       {load_time:.2f}s")
print(f"  Total time:      {extract_time + transform_time + load_time:.2f}s")
print("=" * 60)
